# 02 — GTFS to a LOOM line graph

`gtfs2graph | topo | loom`. Every tool reads a GeoJSON line graph on stdin and
writes one on stdout, so each stage can be inspected on its own.

Needs the LOOM image: `docker build -t openschematicmaps/loom docker/`

In [ ]:
%load_ext autoreload
%autoreload 2

from schematic import feeds, loom, pipeline, animate
from schematic.linegraph import LineGraph
from schematic.crs import to_mercator
from schematic.render import render, octilinearity, Style

FEED = "la-metro-rail"
LINE_ORDER = list("ABCDEK")   # the order lines are drawn in, back to front

In [ ]:
paths = pipeline.schematize(FEED)          # cached; force=True to redo
{stage: p.name for stage, p in paths.items()}

### What each stage does to the graph

`topo` merges the two platforms at an interchange into a single node — which is
why a few GTFS stop_ids will later have no node of their own. `loom` changes no
geometry at all: it solves the left-to-right *ordering* of lines on each edge,
which is what lets the renderer offset parallel tracks correctly.

In [ ]:
for stage, path in paths.items():
    g = LineGraph.from_geojson(path)
    m = g.reproject(to_mercator)
    ok, total = octilinearity(m)
    print(f"{stage:12} {g.summary()}")
    print(f"{'':12} octilinear: {100*ok/total:.1f}% of length")

### Spike: does station identity survive the pipeline?

Everything downstream depends on `station_id` still being the GTFS stop_id after
`octi`. If this ever prints False, the animation has no way to attach a
timetable to the map.

In [ ]:
g = LineGraph.from_geojson(paths["octi"])
print("all stations keep an id:", all(n.station_id for n in g.stations))
print("all stations keep a label:", all(n.station_label for n in g.stations))
[(n.station_id, n.station_label) for n in g.stations[:5]]

### Shared trunks — the edges that need parallel tracks

In [ ]:
shared = [e for e in g.edges if len(e.lines) > 1]
print(f"{len(shared)} of {len(g.edges)} edges carry more than one line")
for e in shared:
    print(f"  {e.line_labels}  {g.nodes[e.src].station_label} -> {g.nodes[e.dst].station_label}")